# Demo for Splitted model

This notebook demostrates

- fine-grained phoneme duration control
- noise control in flow
- streaming decoding in wav decoder

In the splitted model

In [ ]:
from pathlib import Path

import torch

from style_bert_vits2.tts_model import TTSModel


config_path = Path("model_assets/jvnv-F1-jp/config.json")
model_path = Path("model_assets/jvnv-F1-jp/jvnv-F1-jp_e160_s14000.safetensors")
style_vec_path = Path("model_assets/jvnv-F1-jp/style_vectors.npy")

device = "cpu"

tts_model = TTSModel(
    model_path=model_path,
    config_path=config_path,
    style_vec_path=style_vec_path,
    device=device,
)
tts_model.load()

### Biolerplate

In [ ]:
from style_bert_vits2.constants import (
    DEFAULT_ASSIST_TEXT_WEIGHT,
    DEFAULT_STYLE_WEIGHT,
    Languages,
)
from style_bert_vits2.models.infer import get_text


with torch.no_grad():
    bert, ja_bert, en_bert, phones, tones, lang_ids = get_text(
        "今日はいい天気ですね。",
        Languages.JP,
        tts_model.hyper_parameters,
        device,
        assist_text=None,
        assist_text_weight=DEFAULT_ASSIST_TEXT_WEIGHT,
        given_phone=None,
        given_tone=None,
    )
    style_id = 0
    # スタイルベクトルを取得
    style_vector = tts_model.get_style_vector(style_id, DEFAULT_STYLE_WEIGHT)

    # モデルの入力を作成
    x_tst = phones.to(device).unsqueeze(0)
    tones = tones.to(device).unsqueeze(0)
    lang_ids = lang_ids.to(device).unsqueeze(0)
    bert = bert.to(device).unsqueeze(0)
    ja_bert = ja_bert.to(device).unsqueeze(0)
    en_bert = en_bert.to(device).unsqueeze(0)
    x_tst_lengths = torch.LongTensor([phones.size(0)]).to(device)
    style_vec_tensor = torch.from_numpy(style_vector).to(device).unsqueeze(0)
    sid = 0
    sid_tensor = torch.LongTensor([sid]).to(device)
    length_scale = torch.tensor(1.0)
    sdp_ratio = torch.tensor(0.0)
    noise_scale = torch.tensor(0.667)
    noise_scale_w = torch.tensor(0.8)

In [ ]:
from style_bert_vits2.models.models_jp_extra import (
    SynthesizerTrn as SynthesizerTrnJPExtra,
)


# `split` is only implemented in JPExtra for now
assert isinstance(tts_model.net_g, SynthesizerTrnJPExtra)
spk_emb, text_dur, flow, dec = tts_model.net_g.split()  # type: ignore

In [ ]:
with torch.no_grad():
    g = spk_emb.forward(sid_tensor)
    m_p, logs_p, x_mask, w_ceil = text_dur.forward(
        x_tst,
        x_tst_lengths,
        g,
        tones,
        lang_ids,
        ja_bert,
        style_vec_tensor,
        length_scale,
        noise_scale_w,
        sdp_ratio,
    )
    # the necessary evil of pre-calculating the shape of `z`
    # TODO: may be handle later
    B,N,T = m_p.shape
    max_len = int(w_ceil.sum(-1).max().item())
    eps = torch.rand([B,N,max_len], device=m_p.device)
    z = flow.forward(
        m_p,
        logs_p,
        w_ceil,
        g,
        x_mask,
        eps,
        noise_scale,
    )
    wav = dec.forward(g, z, max_len=None)

In [ ]:
from IPython.display import Audio


def display_audio(wav: torch.Tensor):
    return Audio(wav.squeeze().detach().numpy(), rate=44100)

display_audio(wav)

### Phoneme duration control

In [ ]:
from style_bert_vits2.nlp.symbols import SYMBOLS


# check each phoneme's position
"|".join([f"{idx}:{SYMBOLS[i]}" for idx, i in enumerate(phones)])

We'll strech the `テ` part `(17-20)`

In [ ]:
new_w = torch.clone(w_ceil)
new_w[:,:,17] = new_w[:,:,17] * 3.0
new_w[:,:,18] = new_w[:,:,18] * 3.0
new_w[:,:,19] = new_w[:,:,19] * 3.0
new_w[:,:,20] = new_w[:,:,20] * 3.0
print(f"Before:\t{w_ceil.flatten().tolist()}")
print(f"After:\t{new_w.flatten().tolist()}")

In [ ]:
# inference with modified duration
noise = torch.rand_like(m_p)
with torch.no_grad():
    B,N,T = m_p.shape
    # need to calculate the new eps shape as we changed the total length
    # we'll make this neater afterwards.
    max_len = int(new_w.sum(-1).max().item())
    eps = torch.rand([B,N,max_len], device=m_p.device)
    new_z = flow.forward(
        m_p,
        logs_p,
        new_w,
        g,
        x_mask,
        eps,
        noise_scale,
    )
    max_mu_len = torch.sum(new_w)
    new_wav = dec.forward(g, new_z, max_len=None)

display_audio(new_wav)

### Noise control

We just need to change the seed in generation of `eps`.

There are 2 points to insert randomness:

- TextDur: insert in duration predictor
- Flow: insert in variational inference

Only the latter one is implemented for now.

In [ ]:
def get_wav(seed: int):
    with torch.no_grad():
        B,N,_ = m_p.shape
        max_len = int(w_ceil.sum(-1).max().item())
        # use the fixed generator
        eps = torch.rand([B,N,max_len], device=m_p.device, generator=torch.Generator().manual_seed(seed))
        z = flow.forward(
            m_p,
            logs_p,
            w_ceil,
            g,
            x_mask,
            eps,
            noise_scale,
        )
        wav = dec.forward(g, z, max_len=None)
    return wav

#### Check how much the wav has changed with seed

Despite being numerically significant, the change is barely hearable for human ears.

In [ ]:
wav_1 = get_wav(3407)
wav_2 = get_wav(3407)

torch.all((wav_1 - wav_2) < 1e-6)

In [ ]:
wav_1 = get_wav(4935)
wav_2 = get_wav(3407)

torch.all((wav_1 - wav_2) < 1e-6)

In [ ]:
import matplotlib.pyplot as plt


plt.plot(wav_1.detach().cpu().squeeze(), color="blue")
plt.plot(wav_2.detach().cpu().squeeze(), color="green")

plt.show()

In [ ]:
from IPython.display import display


# it's pretty difficult tell the difference
display(
    display_audio(wav_1),
    display_audio(wav_2)
)

TODO: insert eps in `sdp`

### Streaming decoding

Demostrated in a sync for loop, I don't like async, like, really.

In [ ]:
import time


# use a longer text for better comparation
with torch.no_grad():
    bert, ja_bert, en_bert, phones, tones, lang_ids = get_text(
        "貴下の哀愁感が、もし将来に於いて哲学的に整理できたならば、貴下の小説も今日の如く嘲笑せられず、貴下の人格も完成される事と存じます。",
        Languages.JP,
        tts_model.hyper_parameters,
        device,
        assist_text=None,
        assist_text_weight=DEFAULT_ASSIST_TEXT_WEIGHT,
        given_phone=None,
        given_tone=None,
    )
    style_id = 0
    # スタイルベクトルを取得
    style_vector = tts_model.get_style_vector(style_id, DEFAULT_STYLE_WEIGHT)

    # モデルの入力を作成
    x_tst = phones.to(device).unsqueeze(0)
    tones = tones.to(device).unsqueeze(0)
    lang_ids = lang_ids.to(device).unsqueeze(0)
    bert = bert.to(device).unsqueeze(0)
    ja_bert = ja_bert.to(device).unsqueeze(0)
    en_bert = en_bert.to(device).unsqueeze(0)
    x_tst_lengths = torch.LongTensor([phones.size(0)]).to(device)
    style_vec_tensor = torch.from_numpy(style_vector).to(device).unsqueeze(0)
    sid = 0
    sid_tensor = torch.LongTensor([sid]).to(device)
    length_scale = torch.tensor(1.0)
    sdp_ratio = torch.tensor(0.0)
    noise_scale = torch.tensor(0.667)
    noise_scale_w = torch.tensor(0.8)

    g = spk_emb.forward(sid_tensor)
    m_p, logs_p, x_mask, w_ceil = text_dur.forward(
        x_tst,
        x_tst_lengths,
        g,
        tones,
        lang_ids,
        ja_bert,
        style_vec_tensor,
        length_scale,
        noise_scale_w,
        sdp_ratio,
    )
    B, N, T = m_p.shape
    max_len = int(w_ceil.sum(-1).max().item())
    eps = torch.rand([B, N, max_len], device=m_p.device)
    z = flow.forward(
        m_p,
        logs_p,
        w_ceil,
        g,
        x_mask,
        eps,
        noise_scale,
    )
    start = time.time()
    wav = dec.forward(g, z, max_len=None)
    print(f"A full decoding process takes {time.time()-start:.4f} seconds.")

In [ ]:
import torch.nn.functional as F


# ripped off from https://qiita.com/__dAi00/items/970f0fe66286510537dd
kernel_size = 50
padding = 11

hop_length = tts_model.hyper_parameters.data.hop_length

# how much `wav` data points is mapped to 1 frame in `z`
scale_ratio = hop_length

T = z.shape[-1]
n_segments = T // kernel_size + 1

segs = []

with torch.no_grad():
    padded_z = F.pad(z, (padding, padding), value=0.0)
    start = time.time()
    for i in range(n_segments):
        seg_z = padded_z[:, :, i * kernel_size : (i + 1) * kernel_size + 2 * padding]
        seg_wav = dec.forward(g, seg_z, None)
        seg_wav = seg_wav[:, :, scale_ratio * padding : -scale_ratio * padding]
        print(f"New segment, elapsed: {time.time() - start:.4f} second")
        segs.append(seg_wav)

cat_wav = torch.cat(segs, dim=-1)

plt.plot((cat_wav - wav).squeeze().detach().cpu().numpy(), color="blue")
plt.show()

In [ ]:
display(
    display_audio(cat_wav),
    display_audio(wav)
)